# Step 2 — Human baselines and TPR cases

Computes everything on the human side of the TPR benchmark, from the raw CHILDES transcripts:

1. **Manchester TPR cases.** Every child or caretaker D×N site whose noun the other speaker mentioned earlier in the session, with the antecedent's determiner and distance → `output/manchester_tpr_childes/tpr_cached_inputs.csv` (used by Steps 3 and 4).
2. **Human TPR per dyad**, unrestricted noun set (the one reported in the paper), for the 12 Manchester dyads and the 17 tpr-data dyads with other adults.
3. **The adult reference**: other adults responding to mothers in tpr-data, pooled = 0.215 over 1,615 transitions.
4. **Tests** of children and caretakers against that reference, and a check that the two corpora agree.
5. **Antecedent distance**: TPR as a function of how far back the antecedent is (supplementary figures; window-matched references used by Step 4).

Needs `data/CHILDES/Manchester/` and `data/CHILDES/tpr-data/` (see `data/README.md`). Writes to `results/tpr/` and `figures/`.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import ttest_1samp, ttest_ind, ttest_rel

from determiner import tpr
from determiner.corpus import load_manchester_corpus, load_tpr_data_corpus
from determiner.paths import chdir_to_root
from determiner.sites import load_or_extract_det_noun_locations

chdir_to_root()
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

MANCHESTER_OUT = "./output/manchester_tpr_childes"
ADULT_OUT = "./output/tpr_data_analysis"
RESULTS_TPR = "./results/tpr"
for d in (MANCHESTER_OUT, ADULT_OUT, RESULTS_TPR):
    os.makedirs(d, exist_ok=True)

## 1. Manchester: TPR cases

Sites come from the committed inventory (`results/site_inventory/`). A TPR case is a site whose noun was last mentioned, earlier in the same session, by the other speaker; every noun counts.

In [ ]:
manchester_corpus = load_manchester_corpus()
manchester_sites = load_or_extract_det_noun_locations(manchester_corpus, MANCHESTER_OUT)
tpr_cases = tpr.add_antecedent_distance(tpr.prepare_tpr_cases(manchester_sites), manchester_corpus)
tpr_cases.to_csv(os.path.join(MANCHESTER_OUT, "tpr_cached_inputs.csv"), index=False)
print(f"{len(tpr_cases)} TPR cases; antecedent distance median={tpr_cases.true_dist.median():.0f}, "
      f"mean={tpr_cases.true_dist.mean():.1f} utterances")

# Human TPR restricted to nouns both speakers used with both determiners
# (results/tpr/tpr_human_baseline.csv). Kept for reference; the paper uses every noun (§2).
attested_nouns = tpr.find_attested_nouns_per_session(manchester_corpus, manchester_sites)
_ = tpr.calculate_tpr_human_baseline(manchester_corpus, manchester_sites, attested_nouns,
                                     tpr_output_dir=MANCHESTER_OUT, results_tpr_dir=RESULTS_TPR)

## 2. Human TPR per dyad (unrestricted noun set)

Per-session counts are summed within each dyad before dividing, so long sessions weigh more than short ones.

In [ ]:
def per_dyad_tpr(per_session, column):
    """Sum transition counts within each (dyad, speaker), then divide."""
    agg = per_session.groupby(["child_name_norm", "speaker"]).agg({
        "n_transitions_to_the": "sum",
        "n_transitions_to_a": "sum",
        "n_total": "sum",
    }).reset_index()
    agg[column] = (agg["n_transitions_to_the"] + agg["n_transitions_to_a"]) / agg["n_total"]
    return agg[["child_name_norm", "speaker", column]]


# Manchester: 12 child–mother dyads.
manchester_tpr = tpr.calculate_tpr_human_baseline_unrestricted(
    manchester_sites, os.path.join(MANCHESTER_OUT, "tpr_human_baseline_unrestricted.csv"))
manchester_tpr["tpr"] = ((manchester_tpr["n_transitions_to_the"] + manchester_tpr["n_transitions_to_a"])
                         / manchester_tpr["n_total"].replace(0, np.nan))
manchester_tpr.to_csv(os.path.join(RESULTS_TPR, "tpr_human_baseline_unrestricted.csv"), index=False)
manchester_tpr["child_name_norm"] = manchester_tpr["child_name"].str.lower().str.replace("warr", "warren")
manchester_dyads = per_dyad_tpr(manchester_tpr, "tpr")

# tpr-data: 17 dyads that include other adults.
adult_corpus = load_tpr_data_corpus()
adult_sites = load_or_extract_det_noun_locations(adult_corpus, ADULT_OUT)
adult_tpr = tpr.calculate_tpr_human_baseline_unrestricted(
    adult_sites, os.path.join(ADULT_OUT, "tpr_human_baseline_unrestricted.csv"))
adult_tpr["child_name_norm"] = adult_tpr["child_name"].str.lower().str.replace("warr", "warren")
adult_dyads = per_dyad_tpr(adult_tpr, "tpr")

children = manchester_dyads.query("speaker == 'child'")["tpr"].dropna()
caretakers = manchester_dyads.query("speaker == 'mother'")["tpr"].dropna()
other_adults = adult_dyads.query("speaker == 'other_adult'")["tpr"].dropna()

## 3. The adult reference

Other adults responding to the mother in tpr-data, pooled over all transitions (counts summed before dividing).

In [ ]:
oa_counts = adult_tpr.loc[adult_tpr["speaker"] == "other_adult",
                          ["n_transitions_to_the", "n_transitions_to_a", "n_total"]].sum()
ADULT_TPR = (oa_counts["n_transitions_to_the"] + oa_counts["n_transitions_to_a"]) / oa_counts["n_total"]
print(f"Adult reference TPR = {ADULT_TPR:.3f} over {int(oa_counts['n_total']):,} transitions "
      f"({len(other_adults)} dyads; per-dyad mean {other_adults.mean():.3f}, SD {other_adults.std(ddof=1):.3f})")

## 4. Tests

* Children and caretakers vs the adult reference (one-sample *t*; paper §4).
* Children vs their caretakers (paired *t*).
* Welch *t* against the other-adult dyads.
* Corpus check: Manchester children and caretakers vs the tpr-data children and caretakers (Welch *t*), to confirm the two corpora are comparable before tpr-data supplies the reference.

In [ ]:
p_child_validation = ttest_ind(children, adult_dyads.query("speaker == 'child'")["tpr"].dropna(), equal_var=False).pvalue
p_mother_validation = ttest_ind(caretakers, adult_dyads.query("speaker == 'mother'")["tpr"].dropna(), equal_var=False).pvalue
t_child_vs_caretaker, p_child_vs_caretaker = ttest_rel(children, caretakers)


def group_row(speaker, corpus, values, tests):
    return {"speaker": speaker, "corpus": corpus, "n_dyads": int(len(values)),
            "mean_tpr": float(values.mean()), "sd_tpr": float(values.std(ddof=1)),
            **tests, "adult_tpr_population_mean": float(ADULT_TPR)}


no_tests = dict.fromkeys(["p_validation", "p_1sample", "p_welch_vs_adults", "p_child_vs_caretaker"], np.nan)
human_summary = pd.DataFrame([
    group_row("child", "manchester", children, {
        "p_validation": float(p_child_validation),
        "p_1sample": float(ttest_1samp(children, ADULT_TPR).pvalue),
        "p_welch_vs_adults": float(ttest_ind(children, other_adults, equal_var=False).pvalue),
        "p_child_vs_caretaker": float(p_child_vs_caretaker)}),
    group_row("mother", "manchester", caretakers, {
        "p_validation": float(p_mother_validation),
        "p_1sample": float(ttest_1samp(caretakers, ADULT_TPR).pvalue),
        "p_welch_vs_adults": float(ttest_ind(caretakers, other_adults, equal_var=False).pvalue),
        "p_child_vs_caretaker": float(p_child_vs_caretaker)}),
    group_row("other_adult", "tpr_data", other_adults, no_tests),
])
human_summary.to_csv(os.path.join(RESULTS_TPR, "tpr_human_summary.csv"), index=False)

for name, values in [("children", children), ("caretakers", caretakers)]:
    t, p = ttest_1samp(values, ADULT_TPR)
    print(f"{name:<10} TPR {values.mean():.3f} (SD {values.std(ddof=1):.3f}) vs adult {ADULT_TPR:.3f}: t={t:+.3f}, p={p:.3f}")
print(f"children vs caretakers (paired): t={t_child_vs_caretaker:+.3f}, p={p_child_vs_caretaker:.3f}")
print(f"corpus check (Welch, Manchester vs tpr-data): children p={p_child_validation:.3f}, "
      f"caretakers p={p_mother_validation:.3f}")
human_summary.round(4)

In [ ]:
# Per-dyad reference TPRs used by Step 4 for the per-model tests.
refs = pd.concat([
    manchester_dyads.query("speaker in ['child', 'mother']"),
    adult_dyads.query("speaker == 'other_adult'"),
]).dropna(subset=["tpr"]).rename(columns={"tpr": "tpr_unrestricted"})
refs = refs[["speaker", "child_name_norm", "tpr_unrestricted"]]
refs.to_csv(os.path.join(RESULTS_TPR, "tpr_human_perdyad_references.csv"), index=False)
refs.speaker.value_counts()

## 5. Antecedent distance (supplementary)

TPR is a switch rate, and switching becomes more likely the further back the antecedent is. Each group's TPR is therefore a mixture set by its distribution of antecedent distances. Distances are counted in utterances within the session (`true_dist`). Step 4 uses the window-matched references exported here to compare context-limited models fairly.

In [ ]:
GROUP_COLORS = {"children": "#1f77b4", "caretakers": "#d62728", "other_adult": "#2ca02c"}


def per_dyad_switch(cases):
    return cases.groupby("child_name").switch.mean().mean()


# Children and caretakers: the Manchester TPR cases.
cases = tpr_cases[["child_name", "speaker", "previous_det", "original_det", "true_dist"]].copy()
cases["d_A"] = cases.previous_det.str.lower().map({"a": 0, "an": 0, "the": 1})
cases = cases.dropna(subset=["d_A", "true_dist"])
cases["d_A"] = cases.d_A.astype(int)
cases["switch"] = ((cases.original_det.str.lower() == "the").astype(int) != cases.d_A).astype(int)
manchester_dist = (cases[cases.speaker.isin(["child", "mother"])][["speaker", "child_name", "true_dist", "switch"]]
                   .rename(columns={"speaker": "group", "true_dist": "dist"}))
manchester_dist["group"] = manchester_dist.group.map({"child": "children", "mother": "caretakers"})

# Other adults: distance from each tpr-data other-adult site back to the mother's mention.
session_lines = {(c, fn): np.sort(g.line_num.values)
                 for c, df in adult_corpus.items() for fn, g in df.groupby("filename")}
records = []
for (child, fn), session in adult_sites.groupby(["child_name", "filename"], sort=False):
    lines = session_lines.get((child, fn))
    if lines is None:
        continue
    last = {}
    for site in session.sort_values("line_num").itertuples(index=False):
        det = "the" if site.det == "the" else "a"
        if site.noun in last:
            prev_det, prev_speaker, prev_line = last[site.noun]
            if site.speaker_type == "other_adult" and prev_speaker in tpr.ANTECEDENT_SPEAKERS["other_adult"]:
                dist = int(((lines > prev_line) & (lines < site.line_num)).sum()) + 1
                records.append((child, dist, int(det != prev_det)))
        last[site.noun] = (det, site.speaker_type, site.line_num)
adult_dist = pd.DataFrame(records, columns=["child_name", "dist", "switch"]).assign(group="other_adult")

human_dist = pd.concat([manchester_dist, adult_dist[["group", "child_name", "dist", "switch"]]], ignore_index=True)
print("cases per group:", human_dist.groupby("group").size().to_dict())
print("baselines reproduced -> children %.3f | caretakers %.3f | other adults per-dyad %.3f, pooled %.3f"
      % (per_dyad_switch(human_dist[human_dist.group == "children"]),
         per_dyad_switch(human_dist[human_dist.group == "caretakers"]),
         per_dyad_switch(human_dist[human_dist.group == "other_adult"]), adult_dist.switch.mean()))

### TPR vs the allowed context window *K*

Children and caretakers as per-dyad means; other adults pooled (their full-range average is the 0.215 reference). The table repeats the one-sample tests at each *K*; *K* = all reproduces §4.

In [ ]:
def tpr_within(group, k):
    s = human_dist[(human_dist.group == group) & (human_dist.dist <= k)]
    return s.switch.mean() if group == "other_adult" else s.groupby("child_name").switch.mean().mean()


XMAX = 300
Ks = np.unique(np.linspace(1, XMAX, 150).astype(int))
fig, ax = plt.subplots(figsize=(9.5, 5.5))
for group, color in GROUP_COLORS.items():
    full = tpr_within(group, 10**9)
    label = "pooled" if group == "other_adult" else "per-dyad"
    ax.plot(Ks, [tpr_within(group, k) for k in Ks], "-", color=color, lw=2, label=f"{group} ({label} {full:.3f})")
    ax.axhline(full, color=color, ls=(":" if group == "children" else "--"), lw=(2.2 if group == "children" else 1.4),
               alpha=0.85, zorder=(3 if group == "children" else 2))
ax.axvline(50, color="gray", ls=":", lw=1, label="K = 50")
ax.set_xlim(0, XMAX); ax.set_ylim(0, None)
ax.set_xlabel("context window K = max antecedent distance allowed (utterances)")
ax.set_ylabel("TPR over cases with antecedent ≤ K")
ax.set_title("Human TPR vs allowed context window\n(children/caretakers per-dyad; other-adults pooled = 0.215; distances extend to ~3,700)")
ax.legend(fontsize=8); plt.tight_layout()
plt.savefig("figures/human_tpr_vs_window.png", dpi=130, bbox_inches="tight"); plt.show()

In [ ]:
def per_dyad_values(group, k):
    return human_dist[(human_dist.group == group) & (human_dist.dist <= k)].groupby("child_name").switch.mean().dropna()


rows = []
for k in [1, 3, 5, 10, 20, 50, 100, 10**9]:
    oa = human_dist[(human_dist.group == "other_adult") & (human_dist.dist <= k)]
    oa_ref = oa.switch.mean()
    cv, mv = per_dyad_values("children", k), per_dyad_values("caretakers", k)
    ct, cp = ttest_1samp(cv, oa_ref)
    mt, mp = ttest_1samp(mv, oa_ref)
    ix = cv.index.intersection(mv.index)
    cc_t, cc_p = ttest_rel(cv.loc[ix], mv.loc[ix])
    rows.append(dict(
        K=("all" if k > 10**8 else str(k)),
        child_TPR=round(cv.mean(), 3), child_n=int(((human_dist.group == "children") & (human_dist.dist <= k)).sum()),
        child_t=round(ct, 3), child_p=round(cp, 3),
        caretaker_TPR=round(mv.mean(), 3), caretaker_n=int(((human_dist.group == "caretakers") & (human_dist.dist <= k)).sum()),
        caretaker_t=round(mt, 3), caretaker_p=round(mp, 3), cc_t=round(cc_t, 3), cc_p=round(cc_p, 3),
        otherAdult_pooledTPR=round(oa_ref, 3), otherAdult_n=len(oa)))
human_switch_by_window = pd.DataFrame(rows)
human_switch_by_window.to_csv("results/tpr/human_switch_rate_by_window.csv", index=False)
print(human_switch_by_window.to_string(index=False))

# Window-matched references for Step 4: children per-dyad and other adults pooled, for dist <= K.
window_rows = []
for k in list(range(1, 201)) + [10**9]:
    cc = human_dist[(human_dist.group == "children") & (human_dist.dist <= k)].groupby("child_name").switch.mean()
    oa = human_dist[(human_dist.group == "other_adult") & (human_dist.dist <= k)]
    window_rows.append(dict(K=(10**9 if k > 10**8 else k), children_perdyad_TPR=cc.mean(),
                            other_adult_pooled_TPR=oa.switch.mean(), other_adult_n=len(oa)))
pd.DataFrame(window_rows).to_csv("results/tpr/human_tpr_by_window.csv", index=False)

### Distribution of antecedent distances

In [ ]:
mx = int(human_dist.dist.max())
edges = [0, 1, 2, 5, 10, 20, 50, 100] + list(range(150, mx + 50, 50))
labels = [(f"{b}" if b - a == 1 else f"{a + 1}-{b}") for a, b in zip(edges[:-1], edges[1:])]
x = np.arange(len(labels))
fig, ax = plt.subplots(figsize=(16, 5))
for group, color in GROUP_COLORS.items():
    cnt = (pd.cut(human_dist[human_dist.group == group].dist, bins=edges, labels=labels, right=True)
           .value_counts().reindex(labels).fillna(0))
    ax.plot(x, cnt.values, "-", color=color, lw=1.4, marker="o", ms=2.5, label=group)
ax.axvline(6.5, color="gray", ls=":", lw=1)
tick_i = list(range(8)) + list(range(8, len(labels), 4))
ax.set_xticks(tick_i); ax.set_xticklabels([labels[i] for i in tick_i], rotation=60, ha="right", fontsize=6)
ax.set_xlabel("antecedent distance (utterances; ≤10-wide early bins, 50-wide tail bins)")
ax.set_ylabel("# TPR cases"); ax.set_title("Antecedent-distance distribution — all groups (dotted = end of 51-100 bucket)")
ax.legend(); plt.tight_layout()
plt.savefig("figures/human_antecedent_distance_hist.png", dpi=130, bbox_inches="tight"); plt.show()

counts = pd.DataFrame({group: pd.cut(human_dist[human_dist.group == group].dist, bins=edges, labels=labels, right=True)
                       .value_counts().reindex(labels) for group in GROUP_COLORS})
counts["all"] = counts.sum(axis=1)
counts.to_csv("results/tpr/human_antecedent_distance_counts.csv")
pct = pd.DataFrame({group: np.percentile(human_dist[human_dist.group == group].dist, [50, 75, 90, 95, 99, 100])
                    for group in GROUP_COLORS}, index=["p50", "p75", "p90", "p95", "p99", "max"]).astype(int)
print("percentiles of antecedent distance:\n", pct.to_string())

rows = []
for group in GROUP_COLORS:
    s = human_dist[human_dist.group == group]
    rows.append(dict(group=group, n=len(s), median_dist=int(s.dist.median()), pct_gt50=round(100 * (s.dist > 50).mean()),
                     TPR_all=round(per_dyad_switch(s), 3), TPR_le50=round(per_dyad_switch(s[s.dist <= 50]), 3),
                     TPR_gt50=round(per_dyad_switch(s[s.dist > 50]), 3)))
tpr_by_distance = pd.DataFrame(rows)
tpr_by_distance.to_csv("results/tpr/human_tpr_by_distance_summary.csv", index=False)
print("\nTPR by near/far antecedent:\n", tpr_by_distance.to_string(index=False))

Switching is rare for recent antecedents (about 0.07–0.17 within 50 utterances) and roughly doubles for far ones. The other-adult reference is dominated by far antecedents (median about 56 utterances, about half beyond 50), which is what lifts its pooled TPR to 0.215; children's and caretakers' antecedents are more recent.